In [1]:
from pynq import Overlay, allocate
import numpy as np
import time
from pathlib import Path

print("Imports successful")

Imports successful


In [14]:
try: 
    bnn_overlay = Overlay('/home/xilinx/jupyter_notebooks/MNIST/vivado/bnn_top.bit')
    print("Bitstream loaded successfully!")
    print(f"\nOverlay contains: {overlay.ip_dict.keys()}")
except Exception as e:
    print(f"Error loading bitstream: {e}")
    print("\nMake sure bnn_top.bit and bnn_top.hwh are in the same directory!")

Bitstream loaded successfully!

Overlay contains: dict_keys(['bnn_top_0', 'axi_dma', 'zynq_ultra_ps_e_0'])


In [15]:
dma = bnn_overlay.axi_dma
dma_send = bnn_overlay.axi_dma.sendchannel
dma_recv = bnn_overlay.axi_dma.recvchannel
bnn_ip = bnn_overlay.bnn_top_0

In [16]:
def pack_binary_input(binary_array):
    """
    Pack 784-element binary array into ap_uint<784> representation
    
    Args:
        binary_array: numpy array of 0/1 values, shape (784,)
    
    Returns:
        Integer representing packed 784 bits
    """
    packed = 0
    for i in range(len(binary_array)):
        if binary_array[i] == 1:
            packed |= (1 << i)
    return packed


def unpack_output(output_buffer, num_neurons=256):
    """
    Unpack output buffer from FPGA
    
    Args:
        output_buffer: Memory buffer or list of values
        num_neurons: Number of output neurons
    
    Returns:
        numpy array of signed integers
    """
    output = np.zeros(num_neurons, dtype=np.int32)
    
    if isinstance(output_buffer, np.ndarray):
        output[:] = output_buffer[:num_neurons]
    else:
        for i in range(num_neurons):
            output[i] = output_buffer[i]
    
    return output


def compute_expected_output(input_binary, weights):
    """
    Software reference implementation for verification
    
    Args:
        input_binary: 784-element array of {0, 1}
        weights: (256, 784) array of {-1, +1}
    
    Returns:
        256-element array of expected outputs
    """
    num_neurons = weights.shape[0]
    output = np.zeros(num_neurons, dtype=np.int32)
    
    for n in range(num_neurons):
        # Pack weights for this neuron
        weight_bits = (weights[n, :] < 0).astype(np.uint8)
        
        # XNOR: matching bits
        xnor_result = ~(input_binary ^ weight_bits) & 1
        
        # Popcount
        count = np.sum(xnor_result)
        
        # Bipolar conversion
        output[n] = 2 * count - 784
    
    return output

print("Helper functions defined")


Helper functions defined


In [17]:
print("Calling FPGA accelerator...")


try:
    # Allocate output buffer in contiguous memory
    output_buffer = allocate(shape=(256,), dtype=np.int32)
    
    # Option A: If your IP has .call() method
    if hasattr(bnn_ip, 'call'):
        print("Using .call() method...")
        bnn_ip.call(input_packed, output_buffer)
    
    # Option B: Manual register access
    else:
        print("Using manual register access...")
        
        # Write input (address depends on your IP)
        # This is example - adjust based on your register map
        bnn_ip.write(0x10, input_packed & 0xFFFFFFFF)  # Lower 32 bits
        # If input > 32 bits, need multiple writes for full 784 bits
        
        # Write output buffer address
        bnn_ip.write(0x18, output_buffer.physical_address)
        
        # Start computation
        bnn_ip.write(0x00, 0x01)  # AP_START
        
        # Wait for completion
        timeout = 1000  # milliseconds
        start_time = time.time()
        
        while (bnn_ip.read(0x00) & 0x02) == 0:  # Wait for AP_DONE
            if (time.time() - start_time) * 1000 > timeout:
                print("Timeout waiting for FPGA!")
                break
            time.sleep(0.001)
        
        if (bnn_ip.read(0x00) & 0x02):
            print("FPGA computation complete")
    
    # Read results
    fpga_output = output_buffer.copy()
    
    print(f"\nFPGA output[0]: {fpga_output[0]}")
    print(f"Expected:       {expected_output[0]}")
    print(f"Match: {'PASS ✓' if fpga_output[0] == expected_output[0] else 'FAIL ✗'}")
    
except Exception as e:
    print(f"Error during FPGA execution: {e}")
    import traceback
    traceback.print_exc()

Calling FPGA accelerator...
Using manual register access...
Error during FPGA execution: name 'input_packed' is not defined


Traceback (most recent call last):
  File "/tmp/ipykernel_949/1938941227.py", line 19, in <cell line: 4>
    bnn_ip.write(0x10, input_packed & 0xFFFFFFFF)  # Lower 32 bits
NameError: name 'input_packed' is not defined


In [10]:
try:
    from torchvision import datasets, transforms
    
    # Load MNIST
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    test_dataset = datasets.MNIST(
        '../phase1_software/data', 
        train=False, 
        download=True, 
        transform=transform
    )
    
    print(f"Loaded MNIST test set: {len(test_dataset)} images")
    
except Exception as e:
    print(f"Error loading MNIST: {e}")
    print("Install torchvision: pip install torchvision")
    test_dataset = None

Error loading MNIST: No module named 'torchvision'
Install torchvision: pip install torchvision


In [11]:
if test_dataset is not None:
    print("MNIST Images Testbench")

    num_test_images = 10
    results = []

    for idx in range(num_test_images):
        # Get MNIST image
        image, label = test_dataset[idx]
        
        # Flatten and binarize
        image_flat = image.view(-1).numpy()  # 784 values
        image_binary = (image_flat > 0).astype(np.uint8)  # Threshold at 0
        
        # Expected output (software)
        expected = compute_expected_output(image_binary, layer1_weights)
        
        # Pack and send to FPGA
        input_packed_test = pack_binary_input(image_binary)
        output_buffer_test = allocate(shape=(256,), dtype=np.int32)
        
        # Call FPGA
        if hasattr(bnn_ip, 'call'):
            bnn_ip.call(input_packed_test, output_buffer_test)
        else:
            bnn_ip.write(0x10, input_packed_test & 0xFFFFFFFF)
            bnn_ip.write(0x18, output_buffer_test.physical_address)
            bnn_ip.write(0x00, 0x01)
            while (bnn_ip.read(0x00) & 0x02) == 0:
                time.sleep(0.001)
        
        fpga_out = output_buffer_test.copy()
        
        # Compare
        match = np.array_equal(fpga_out, expected)
        results.append(match)
        
        print(f"Image {idx} (label={label}): {'PASS ✓' if match else 'FAIL ✗'}")
        if not match:
            diff = np.sum(fpga_out != expected)
            print(f"  {diff} / 256 outputs differ")
        
        # Free buffer
        del output_buffer_test

    print(f"\nOverall: {sum(results)} / {num_test_images} tests passed")
else:
    print("\nDataset not loaded")
    results = []



Dataset not loaded


In [12]:

print("PERFORMANCE BENCHMARK")

num_iterations = 100
test_input_perf = np.zeros(784, dtype=np.uint8)
input_packed_perf = pack_binary_input(test_input_perf)
output_buffer_perf = allocate(shape=(256,), dtype=np.int32)

# Warm-up
print("Running warm-up...")
for _ in range(10):
    if hasattr(bnn_ip, 'call'):
        bnn_ip.call(input_packed_perf, output_buffer_perf)
    else:
        bnn_ip.write(0x10, input_packed_perf & 0xFFFFFFFF)
        bnn_ip.write(0x18, output_buffer_perf.physical_address)
        bnn_ip.write(0x00, 0x01)
        while (bnn_ip.read(0x00) & 0x02) == 0:
            pass

print("Warm-up complete\n")
print(f"Running {num_iterations} iterations...")

# Benchmark
times = []

for i in range(num_iterations):
    start = time.perf_counter()
    
    if hasattr(bnn_ip, 'call'):
        bnn_ip.call(input_packed_perf, output_buffer_perf)
    else:
        bnn_ip.write(0x10, input_packed_perf & 0xFFFFFFFF)
        bnn_ip.write(0x18, output_buffer_perf.physical_address)
        bnn_ip.write(0x00, 0x01)
        while (bnn_ip.read(0x00) & 0x02) == 0:
            pass
    
    end = time.perf_counter()
    times.append((end - start) * 1000)  # Convert to ms

times = np.array(times)

print("\nResults:")
print(f"  Average latency:  {np.mean(times):.4f} ms")
print(f"  Min latency:      {np.min(times):.4f} ms")
print(f"  Max latency:      {np.max(times):.4f} ms")
print(f"  Std deviation:    {np.std(times):.4f} ms")
print(f"\n  Throughput:       {1000/np.mean(times):.0f} images/second")


PERFORMANCE BENCHMARK
Running warm-up...


NameError: name 'bnn_ip' is not defined